# Árvore Heap e Heap Sort (Python)

Este notebook apresenta a estrutura **heap** e o algoritmo **heap sort** de forma didática.

Objetivos:
- Entender o que é uma heap (max-heap e min-heap).
- Representar uma heap em vetor com índice 0 vazio.
- Implementar `heapify` e construção de max-heap.
- Implementar `heap_sort` passo a passo.
- Validar com exemplos e testes simples.

In [ ]:
# Se necessário para reprodutibilidade dos exemplos
import random
import time

print("Notebook carregado com sucesso.")

## 1) Conceito de Heap

Uma **heap binária** é uma árvore binária completa que respeita uma propriedade de ordem:
- **Max-heap**: cada pai é maior ou igual aos filhos.
- **Min-heap**: cada pai é menor ou igual aos filhos.

Neste notebook, usaremos **max-heap** para implementar heap sort.

Neste material, usaremos heap com **indexação iniciando em 1**.
Isso significa que a posição `0` do vetor fica vazia (ex.: `None`).

Representação em vetor (índice `i`):
- Filho esquerdo: `2*i`
- Filho direito: `2*i + 1`
- Pai: `i // 2`

In [ ]:
def filhos(i):
    return 2 * i, 2 * i + 1


def pai(i):
    return i // 2 if i > 1 else None


for i in range(1, 6):
    print(f"índice {i} -> pai {pai(i)}, filhos {filhos(i)}")

## 2) Função `heapify` (max-heap)

A função `heapify_max` garante a propriedade de max-heap para um nó `i`,
assumindo que as subárvores dos filhos já são heaps válidas.

Complexidade de uma chamada: $O(\log n)$ no pior caso.

In [ ]:
def heapify_max(v, n, i):
    """
    Ajusta a subárvore enraizada em i para obedecer à propriedade de max-heap.
    v: vetor com posição 0 vazia (indexação iniciando em 1)
    n: último índice válido da heap no vetor
    i: índice da raiz da subárvore
    """
    maior = i
    esq = 2 * i
    dir = 2 * i + 1

    if esq <= n and v[esq] > v[maior]:
        maior = esq

    if dir <= n and v[dir] > v[maior]:
        maior = dir

    if maior != i:
        v[i], v[maior] = v[maior], v[i]
        heapify_max(v, n, maior)


exemplo = [None, 10, 3, 8, 2, 1, 4]
print("Antes do heapify na raiz:", exemplo)
heapify_max(exemplo, len(exemplo) - 1, 1)
print("Depois do heapify na raiz:", exemplo)

## 3) Construindo uma Max-Heap

Para transformar um vetor (com posição 0 vazia) em max-heap, aplicamos `heapify_max`
de baixo para cima, iniciando no último nó interno.

Complexidade total da construção: $O(n)$.

In [ ]:
def construir_max_heap(v):
    n = len(v) - 1
    # Último nó interno (indexação em 1): n // 2
    for i in range(n // 2, 0, -1):
        heapify_max(v, n, i)


dados = [None, 12, 11, 13, 5, 6, 7, 20, 1]
print("Vetor original:", dados)
construir_max_heap(dados)
print("Após construir max-heap:", dados)

## 3.1) Abordagens top-down e bottom-up

Existem duas formas clássicas de construir uma heap:

- **Top-down (de cima para baixo)**: inserimos os elementos um a um e fazemos o elemento "subir" (`sift-up`) até a posição correta.
- **Bottom-up (de baixo para cima)**: partimos de um vetor completo e aplicamos `heapify` nos nós internos, do fim para o início.

Com indexação iniciando em 1 (posição 0 vazia), as duas produzem o mesmo resultado de heap válida, mas com custos diferentes:

- **Top-down**: $O(n \log n)$
- **Bottom-up (Floyd)**: $O(n)$

Por isso, para **construir heap a partir de um vetor já disponível**, bottom-up costuma ser a melhor escolha.

In [ ]:
def sift_up_max(v, i):
    """Move v[i] para cima enquanto violar a propriedade de max-heap."""
    while i > 1:
        p = i // 2
        if v[p] >= v[i]:
            break
        v[p], v[i] = v[i], v[p]
        i = p


def construir_max_heap_top_down(valores):
    """Constrói max-heap por inserções sucessivas (posição 0 fica vazia)."""
    heap = [None]
    for x in valores:
        heap.append(x)
        sift_up_max(heap, len(heap) - 1)
    return heap


entrada = [12, 11, 13, 5, 6, 7, 20, 1]
heap_td = construir_max_heap_top_down(entrada)
heap_bu = [None] + entrada[:]
construir_max_heap(heap_bu)

print("Top-down  (heap):", heap_td)
print("Bottom-up (heap):", heap_bu)
print("Ambas são heaps válidas?", heap_td[1] == max(entrada) and heap_bu[1] == max(entrada))

## 4) Heap Sort

Passos do heap sort (ordem crescente):
1. Construir max-heap com todos os elementos.
2. Trocar a raiz (maior elemento) com a última posição da heap.
3. Reduzir o tamanho útil da heap em 1.
4. Aplicar `heapify_max` na raiz.
5. Repetir até o tamanho útil ser 1.

Complexidade:
- Tempo: $O(n \log n)$
- Espaço extra: $O(1)$ (ordenação in-place)

In [ ]:
def heap_sort(v):
    n = len(v) - 1

    # 1) Constrói max-heap
    for i in range(n // 2, 0, -1):
        heapify_max(v, n, i)

    # 2) Extrai o máximo repetidamente
    for fim in range(n, 1, -1):
        v[1], v[fim] = v[fim], v[1]
        heapify_max(v, fim - 1, 1)


valores = [None, 4, 10, 3, 5, 1, 8, 7, 2, 9, 6]
print("Antes (ignorando índice 0):", valores[1:])
heap_sort(valores)
print("Depois (ignorando índice 0):", valores[1:])

## 5) Versão didática com rastreio de etapas

Esta versão imprime cada troca principal para facilitar o entendimento visual.

In [ ]:
def heap_sort_com_rastro(v):
    n = len(v) - 1
    print("\n[1] Construindo max-heap")
    for i in range(n // 2, 0, -1):
        heapify_max(v, n, i)
    print("Heap inicial (índices 1..n):", v[1:])

    print("\n[2] Extraindo máximos")
    for fim in range(n, 1, -1):
        v[1], v[fim] = v[fim], v[1]
        print(f"Troca raiz com índice {fim}: {v[1:]}")
        heapify_max(v, fim - 1, 1)
        print(f"Heap restaurada até índice {fim - 1}: {v[1:]}")


amostra = [None, 15, 3, 17, 10, 84, 19, 6, 22, 9]
print("Amostra inicial (ignorando índice 0):", amostra[1:])
heap_sort_com_rastro(amostra)
print("Resultado final (ignorando índice 0):", amostra[1:])

## 6) Testes rápidos

Vamos validar se o heap sort ordena corretamente comparando com `sorted` em vários cenários.

In [ ]:
def validar_heap_sort(quantidade_testes=200, n_max=80, limite_valor=500):
    random.seed(42)

    for _ in range(quantidade_testes):
        n = random.randint(0, n_max)
        vetor = [random.randint(-limite_valor, limite_valor) for _ in range(n)]
        esperado = sorted(vetor)
        heap = [None] + vetor[:]
        heap_sort(heap)
        obtido = heap[1:]

        if obtido != esperado:
            return False, vetor, esperado, obtido

    return True, None, None, None


ok, original, esp, ob = validar_heap_sort()
if ok:
    print("Todos os testes passaram com sucesso.")
else:
    print("Falha encontrada.")
    print("Original:", original)
    print("Esperado:", esp)
    print("Obtido:", ob)

## 7) Comparação de desempenho simples

Comparação didática entre heap sort e sorted do Python (Timsort).

Observação: `sorted` costuma ser mais rápido na prática por ser altamente otimizado em C.

In [ ]:
def benchmark(tamanho=30000):
    random.seed(123)
    base = [random.randint(-10**6, 10**6) for _ in range(tamanho)]

    a = [None] + base[:]
    t0 = time.perf_counter()
    heap_sort(a)
    t1 = time.perf_counter()

    b = base[:]
    t2 = time.perf_counter()
    b = sorted(b)
    t3 = time.perf_counter()

    print(f"heap_sort: {t1 - t0:.6f} s")
    print(f"sorted:    {t3 - t2:.6f} s")
    print("Resultados iguais?", a[1:] == b)


benchmark(20000)

## 8) Resumo final

- Heap é uma árvore binária completa com propriedade de ordem.
- Heap em vetor com índice 0 vazio permite fórmulas simples de pai e filhos.
- `heapify_max` ajusta uma subárvore em $O(\log n)$.
- Construir heap a partir de vetor custa $O(n)$.
- Heap sort ordena in-place em $O(n \log n)$.

Exercício sugerido:
1. Adapte o notebook para **min-heap**.
2. Use a min-heap para obter os `k` menores elementos de um vetor.